# Assignment 1

![](https://media.giphy.com/media/xT9C25UNTwfZuk85WP/giphy-downsized-large.gif)

Remember the rules of ~Fight~ Code Club:
1. ALWAYS DOCUMENT
2. Cite resources that you use (paste links)
3. Include the names people who you worked with
4. Be neat and organized

## Scrape and Clean data

Based on you proposal, scrape or collect your data:

1. One variable must be either: (40 pts)
    1. scraped from the web OR;
    2. collected from an API AND you must create one new variable that is "new" to the best of your knowledge (combination of other variables representing something new).

2. You must have at least 3 variables, but you may include as many as you want into you final dataset. Likely, you will want to include more to make graphs and regressions. (30pts)

3. You must *be able* to run a regression that makes some sense with this data (the regression doesn't have to be a complete model). Briefly describe one regression you would run with your variables. (**DO NOT** run a regression, yet). (15pts)

4. You must have one combined and cleaned dataset (15 pts)

You must submit one python notebook on how you scraped/gathered data from an api, and how you combined and cleaned you data. I should be able to run your code and reproduce your final data set.  

The other variables that you choose to include do not have to be collected by API or webscraped, but you do have to combine the files and clean the dataset with python.

Thus, you must submit:
- Your finalized data set (only one) (note: you may add more variables in the future).
- Your documented python notebook
- Any associated data files needed to produce the final dataset.

You will be evaluated on:
- Completeness of the data
- Quality of the code 
- The creativity of the new variable/webscraped data you gather

Be sure to upload ALL associated files for your code to run. I will run your code from beginning to end - make it easy for me to replicate your code.

In [1]:
import pandas as pd
import numpy as np
import requests
from pprint import pprint
import os

In [2]:
#for accessing all the data as it is split by month
folder_path = r'C:\Users\iakov\OneDrive\Desktop\ECO590\assignment-1-Iakovos2t\590 Bike'

files_2025 = [
    "JC-202501-citibike-tripdata.csv",
    "JC-202502-citibike-tripdata.csv",
    "JC-202503-citibike-tripdata.csv",
    "JC-202504-citibike-tripdata.csv",
    "JC-202505-citibike-tripdata.csv",
    "JC-202506-citibike-tripdata.csv",
    "JC-202507-citibike-tripdata.csv",
    "JC-202508-citibike-tripdata.csv",
    "JC-202509-citibike-tripdata.csv",
    "JC-202510-citibike-tripdata.csv",
    "JC-202511-citibike-tripdata.csv",
    "JC-202512-citibike-tripdata.csv"
]

In [3]:
#reading all the monthly files and stacking them into one dataframe
#(online help for this)
df_list = []

for file_name in files_2025:
    full_path = os.path.join(folder_path, file_name)
    temp_df = pd.read_csv(full_path)
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)

In [4]:
#checks
df.shape

(1002704, 13)

In [5]:
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,40.742677,-74.051789,member
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,40.711130,-74.078900,member
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member


In [6]:
#turning the time columns into datetime and making ride length in minutes
df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])

df["date"] = df["started_at"].dt.date
df["ride_length_min"] = (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60

In [7]:
#getting rid of weird ride lengths that dont make sense, lost bikes extreme times etc 
df = df[(df["ride_length_min"] > 0) & (df["ride_length_min"] < 180)]

df = df[(df["date"] >= pd.to_datetime("2025-01-01").date()) &
        (df["date"] <= pd.to_datetime("2025-12-31").date())]

In [8]:
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,date,ride_length_min
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,40.742677,-74.051789,member,2025-01-16,6.192900
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,40.711130,-74.078900,member,2025-01-31,11.461350
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,2025-01-09,21.377617
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,2025-01-21,22.934333
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,2025-01-30,25.822100


In [9]:
df["member_casual"].value_counts()

member_casual
member    779521
casual    221388
Name: count, dtype: int64

In [10]:
#checking how many i trrimmed
#only 2000 from 1 milion of >3h
df.shape

(1000909, 15)

In [11]:
#grouping rides by day and by rider type
#making a daily dataset with casual rides member rides total rides and casual share
daily_rides = (df.groupby(["date", "member_casual"]).size().unstack(fill_value=0).reset_index())

daily_rides["total_rides"] = daily_rides["casual"] + daily_rides["member"]
daily_rides["casual_share"] = daily_rides["casual"] / daily_rides["total_rides"]

In [12]:
#sanity check that i didnt lose any rides when i grouped the data
og_total = len(df)
ag_total = daily_rides["total_rides"].sum()

print(og_total,ag_total)

1000909 1000909


In [13]:
#adding the average ride length for each day then for members vs casual
daily_avg_length = df.groupby("date")["ride_length_min"].mean().reset_index()
daily_avg_length.rename(columns={"ride_length_min": "avg_ride_length_mins"}, inplace=True)

daily_rides = pd.merge(daily_rides, daily_avg_length, on="date", how="left")

In [14]:
avg_length_by_type = (df.groupby(["date", "member_casual"])["ride_length_min"].mean().unstack(fill_value=0).reset_index())

avg_length_by_type.rename(columns={"casual": "avg_casual_ride_length_mins","member": "avg_member_ride_length_mins"}, inplace=True)

daily_rides = pd.merge(daily_rides, avg_length_by_type, on="date", how="left")

In [15]:
daily_rides.head()

,date,casual,member,total_rides,casual_share,avg_ride_length_mins,avg_casual_ride_length_mins,avg_member_ride_length_mins
0,2025-01-01,346,832,1178,0.293718,8.990864,13.314308,7.192893
1,2025-01-02,220,1489,1709,0.128730,6.826268,8.320846,6.605444
2,2025-01-03,274,1496,1770,0.154802,6.765170,8.799531,6.392566
3,2025-01-04,265,1072,1337,0.198205,7.481250,9.231497,7.048587
4,2025-01-05,217,972,1189,0.182506,6.916327,8.787102,6.498675


In [16]:
#central Jersey City coordinate
latitude = 40.7190
longitude = -74.0500
#pulling daily weather data for jersey city from open meteo (method for api use from their website)
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": "2025-01-01",
    "end_date": "2025-12-31",
    "daily": "temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum",
    "timezone": "America/New_York"
}

response = requests.get(url, params=params)
data = response.json()

In [17]:
weather_df = pd.DataFrame(data["daily"])
weather_df.head()

,time,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum
0,2025-01-01,10.9,3.9,7.4,4.5
1,2025-01-02,5.4,0.3,2.6,0.0
2,2025-01-03,3.2,-1.9,0.4,0.0
3,2025-01-04,-0.1,-2.7,-1.4,0.0
4,2025-01-05,0.3,-3.6,-2.2,0.0


In [18]:
#changing the names
weather_df = weather_df.rename(columns={
    "time": "date",
    "temperature_2m_max": "temp_max",
    "temperature_2m_min": "temp_min",
    "temperature_2m_mean": "temp_mean",
    "precipitation_sum": "precip_mm"
})

weather_df["date"] = pd.to_datetime(weather_df["date"]).dt.date

#making a few rain variables with different cutoffs
weather_df["rain_any"] = (weather_df["precip_mm"] > 0).astype(int)
weather_df["rain_1mm"] = (weather_df["precip_mm"] >= 1).astype(int)
weather_df["rain_5mm"] = (weather_df["precip_mm"] >= 5).astype(int)
weather_df.head()

,date,temp_max,temp_min,temp_mean,precip_mm,rain_any,rain_1mm,rain_5mm
0,2025-01-01,10.9,3.9,7.4,4.5,1,1,0
1,2025-01-02,5.4,0.3,2.6,0.0,0,0,0
2,2025-01-03,3.2,-1.9,0.4,0.0,0,0,0
3,2025-01-04,-0.1,-2.7,-1.4,0.0,0,0,0
4,2025-01-05,0.3,-3.6,-2.2,0.0,0,0,0


In [19]:
weather_df["precip_mm"].describe()

count    365.000000
mean       3.057808
std        6.926873
min        0.000000
25%        0.000000
50%        0.000000
75%        3.000000
max       48.600000
Name: precip_mm, dtype: float64

In [20]:
#merging the daily bike data with the daily weather data
daily_rides = pd.merge(daily_rides, weather_df, on="date", how="left")
daily_rides.head()


,date,casual,member,total_rides,casual_share,avg_ride_length_mins,avg_casual_ride_length_mins,avg_member_ride_length_mins,temp_max,temp_min,temp_mean,precip_mm,rain_any,rain_1mm,rain_5mm
0,2025-01-01,346,832,1178,0.293718,8.990864,13.314308,7.192893,10.9,3.9,7.4,4.5,1,1,0
1,2025-01-02,220,1489,1709,0.128730,6.826268,8.320846,6.605444,5.4,0.3,2.6,0.0,0,0,0
2,2025-01-03,274,1496,1770,0.154802,6.765170,8.799531,6.392566,3.2,-1.9,0.4,0.0,0,0,0
3,2025-01-04,265,1072,1337,0.198205,7.481250,9.231497,7.048587,-0.1,-2.7,-1.4,0.0,0,0,0
4,2025-01-05,217,972,1189,0.182506,6.916327,8.787102,6.498675,0.3,-3.6,-2.2,0.0,0,0,0


In [21]:
daily_rides.shape

(365, 15)

In [22]:
daily_rides

,date,casual,member,total_rides,casual_share,avg_ride_length_mins,avg_casual_ride_length_mins,avg_member_ride_length_mins,temp_max,temp_min,temp_mean,precip_mm,rain_any,rain_1mm,rain_5mm
0,2025-01-01,346,832,1178,0.293718,8.990864,13.314308,7.192893,10.9,3.9,7.4,4.5,1,1,0
1,2025-01-02,220,1489,1709,0.128730,6.826268,8.320846,6.605444,5.4,0.3,2.6,0.0,0,0,0
2,2025-01-03,274,1496,1770,0.154802,6.765170,8.799531,6.392566,3.2,-1.9,0.4,0.0,0,0,0
3,2025-01-04,265,1072,1337,0.198205,7.481250,9.231497,7.048587,-0.1,-2.7,-1.4,0.0,0,0,0
4,2025-01-05,217,972,1189,0.182506,6.916327,8.787102,6.498675,0.3,-3.6,-2.2,0.0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,2025-12-27,47,242,289,0.162630,8.902440,12.620146,8.180407,-1.0,-7.3,-3.6,5.3,1,1,1
361,2025-12-28,111,393,504,0.220238,12.041178,17.201932,10.583560,1.0,-11.6,-5.8,0.9,1,0,0
362,2025-12-29,149,823,972,0.153292,7.159450,8.901307,6.844095,7.3,-1.4,3.5,3.9,1,1,0
363,2025-12-30,187,969,1156,0.161765,7.772919,9.686362,7.403658,-0.2,-3.0,-1.3,0.0,0,0,0


I would estimate a regression where the dependent variable is daily casual share (casual rides divided by total rides). The main independent variable would be daily precipitation, and I would include daily temperature controls such as maximum, minimum, and mean temperature. This regression would test whether rainfall reduces the share of rides taken by casual riders relative to member riders.

I would like to add more controls such as a weekend dummy, holiday dummy also which i have not done yet to this dataset

In [23]:
daily_rides.to_csv('output.csv', index=False)


More explanation on why you do things, the code is somewhat documented, but could be more detailed

Clearly you wrote this  -it's short and relatively efficient. I like the idea.

97